In [ ]:
import pandas as pd
import numpy as np
from functools import reduce

In [ ]:
DATA_DIR = "dataset/2017-2018"

In [6]:
demo = pd.read_sas(
    f"{DATA_DIR}/DEMO_J.xpt",
    format="xport"
)

bmx = pd.read_sas(
    f"{DATA_DIR}/BMX_J.xpt",
    format="xport"
)

bio = pd.read_sas(
    f"{DATA_DIR}/BIOPRO_J.xpt",
    format="xport"
)

hdl = pd.read_sas(
    f"{DATA_DIR}/HDL_J.xpt",
    format="xport"
)

tchol = pd.read_sas(
    f"{DATA_DIR}/TCHOL_J.xpt",
    format="xport"
)

bpx = pd.read_sas(
    f"{DATA_DIR}/BPX_J.xpt",
    format="xport"
)

ghb = pd.read_sas(
    f"{DATA_DIR}/GHB_J.xpt",
    format="xport"
)

In [7]:
demo_selected = demo[
    ["SEQN", "RIDAGEYR", "RIAGENDR"]
]

In [8]:
bmx_selected = bmx[
    ["SEQN", "BMXBMI"]
]

In [9]:
bio_selected = bio[
    [
        "SEQN",
        "LBXSGL",      # Glucose
        "LBXSAL",      # Albumin
        "LBXSATSI",    # ALT
        "LBXSASSI",    # AST
        "LBXSGTSI",    # GGT
        "LBXSCR",      # Creatinine
        "LBXSBU"       # BUN
    ]
]

In [10]:
hdl_selected = hdl[
    ["SEQN", "LBDHDD"]
]

In [11]:
tchol_selected = tchol[
    ["SEQN", "LBXTC"]
]

In [12]:
bpx["systolic_bp"] = bpx[
    ["BPXSY1", "BPXSY2", "BPXSY3", "BPXSY4"]
].mean(axis=1, skipna=True)

bpx["diastolic_bp"] = bpx[
    ["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]
].mean(axis=1, skipna=True)

In [13]:
bpx_selected = bpx[
    ["SEQN", "systolic_bp", "diastolic_bp"]
]

In [17]:
ghb_selected = ghb[
    ["SEQN", "LBXGH"]
]

In [18]:
datasets = [
    demo_selected,
    bmx_selected,
    bio_selected,
    hdl_selected,
    tchol_selected,
    ghb_selected,
    bpx_selected
]

df = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on="SEQN",
        how="inner"
    ),
    datasets
)

In [19]:
print(df.shape)

(6401, 16)


In [20]:
df = df[
    df["RIDAGEYR"] >= 20
].copy()

In [21]:
print(df.shape)

(5265, 16)


In [22]:
df = df.rename(columns={
    "RIDAGEYR": "age",
    "RIAGENDR": "sex",
    "BMXBMI": "bmi",
    "LBXSGL": "glucose",
    "LBXGH": "hba1c",
    "LBXTC": "total_cholesterol",
    "LBDHDD": "hdl",
    "LBXSAL": "albumin",
    "LBXSATSI": "alt",
    "LBXSASSI": "ast",
    "LBXSGTSI": "ggt",
    "LBXSCR": "creatinine",
    "LBXSBU": "bun"
})

In [23]:
missing = df.isna().sum()

missing_percentage = (
    df.isna().mean() * 100
).round(2)

missing_table = pd.DataFrame({
    "Missing_Count": missing,
    "Missing_Percentage": missing_percentage
})

print(missing_table)

                   Missing_Count  Missing_Percentage
SEQN                           0                0.00
age                            0                0.00
sex                            0                0.00
bmi                           90                1.71
glucose                      351                6.67
albumin                      348                6.61
alt                          350                6.65
ast                          363                6.89
ggt                          350                6.65
creatinine                   349                6.63
bun                          351                6.67
hdl                          328                6.23
total_cholesterol            328                6.23
hba1c                        247                4.69
systolic_bp                  266                5.05
diastolic_bp                 266                5.05


In [24]:
features = [
    "sex",
    "bmi",
    "glucose",
    "hba1c",
    "total_cholesterol",
    "hdl",
    "systolic_bp",
    "diastolic_bp",
    "albumin",
    "alt",
    "ast",
    "ggt",
    "creatinine",
    "bun"
]

target = "age"

In [25]:
df_model = df[
    features + [target]
].dropna().copy()

In [26]:
print("Final dataset shape:", df_model.shape)

Final dataset shape: (4609, 15)


In [27]:
print(df_model.isna().sum())

sex                  0
bmi                  0
glucose              0
hba1c                0
total_cholesterol    0
hdl                  0
systolic_bp          0
diastolic_bp         0
albumin              0
alt                  0
ast                  0
ggt                  0
creatinine           0
bun                  0
age                  0
dtype: int64


In [28]:
df_model.to_csv(
    "nhanes_2017_2018_clean.csv",
    index=False
)

In [30]:
print(df_model.shape)
print(df_model.head())
print(df_model.columns)
print(df_model.describe())

(4609, 15)
   sex   bmi  glucose  hba1c  total_cholesterol   hdl  systolic_bp  \
0  2.0  31.7     85.0    6.2              157.0  60.0   200.000000   
3  2.0  23.7    116.0    6.2              209.0  88.0   142.000000   
4  2.0  38.9     96.0    6.3              176.0  65.0   118.666667   
5  1.0  21.3     98.0    5.7              238.0  72.0   101.333333   
7  1.0  23.5     91.0    5.6              184.0  48.0   104.666667   

   diastolic_bp  albumin   alt   ast   ggt  creatinine   bun   age  
0     68.000000      4.4  16.0  20.0  21.0        0.92  11.0  66.0  
3     76.000000      3.9  19.0  21.0  22.0        0.58  16.0  66.0  
4     66.666667      3.7  15.0  17.0  31.0        1.32  20.0  75.0  
5     66.666667      4.0  20.0  23.0  19.0        1.13  14.0  56.0  
7     72.000000      4.3  18.0  18.0  26.0        1.13  22.0  67.0  
Index(['sex', 'bmi', 'glucose', 'hba1c', 'total_cholesterol', 'hdl',
       'systolic_bp', 'diastolic_bp', 'albumin', 'alt', 'ast', 'ggt',
       'creatin